# AI Knowledge Engine — Google Colab

Запустите ячейки сверху вниз. Все программы, база и модели находятся во временной среде Colab. После её отключения данные удаляются. Не загружайте конфиденциальные документы в публично доступную среду. Ответы модели требуют отдельно настроенного провайдера; поиск работает без него.


In [ ]:
import os, subprocess, secrets, zipfile, urllib.request
from pathlib import Path
def run(*args, **kwargs):
    subprocess.run(list(args), check=True, **kwargs)
url = 'https://github.com/mivrichka1/ai-knowledge-engine/raw/refs/heads/main/ake-colab-source.zip'
urllib.request.urlretrieve(url, '/content/ake-source.zip')
with zipfile.ZipFile('/content/ake-source.zip') as z:
    z.extractall('/content/ai-knowledge-engine')
print('Source downloaded to the temporary Colab runtime.')


In [ ]:
run('bash', '-c', 'apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq postgresql postgresql-server-dev-all build-essential python3.12-venv')
run('git', 'clone', '-q', '--depth', '1', '--branch', 'v0.8.0', 'https://github.com/pgvector/pgvector.git', '/content/pgvector')
run('bash', '-c', 'cd /content/pgvector && make -s && make -s install')
run('/usr/bin/python3.12', '-m', 'venv', '/content/ake-python312')
run('/content/ake-python312/bin/pip', 'install', '-q', 'torch==2.15.0', '--index-url', 'https://download.pytorch.org/whl/cpu')
run('/content/ake-python312/bin/pip', 'install', '-q', '-r', '/content/ai-knowledge-engine/backend/requirements.txt')
run('npm', 'ci', '--silent', cwd='/content/ai-knowledge-engine/frontend')
print('Dependencies installed in Colab.')


In [ ]:
os.environ['AKE_DB_PASSWORD'] = secrets.token_hex(16)
os.environ['JWT_SECRET'] = secrets.token_hex(32)
os.environ['DATABASE_URL'] = 'postgresql+asyncpg://ake:' + os.environ['AKE_DB_PASSWORD'] + '@127.0.0.1:5432/ake'
os.environ['DOCUMENT_STORAGE_PATH'] = '/content/ake-documents'
run('service', 'postgresql', 'start')
run('runuser','-u','postgres','--','psql','-v','ON_ERROR_STOP=1','-c', "CREATE ROLE ake LOGIN PASSWORD '" + os.environ['AKE_DB_PASSWORD'] + "'", stdout=subprocess.DEVNULL)
run('runuser','-u','postgres','--','createdb','-O','ake','ake')
run('runuser','-u','postgres','--','psql','-d','ake','-v','ON_ERROR_STOP=1','-c','CREATE EXTENSION IF NOT EXISTS vector')
run('/content/ake-python312/bin/alembic','upgrade','head',cwd='/content/ai-knowledge-engine/backend')
print('Temporary database is ready.')


In [ ]:
import time
front = Path('/content/ai-knowledge-engine/frontend')
(front/'vite.config.ts').write_text("import { defineConfig } from 'vite';\nimport react from '@vitejs/plugin-react';\nexport default defineConfig({plugins:[react()],server:{host:'127.0.0.1',allowedHosts:true,hmr:false,proxy:{'/api':'http://127.0.0.1:8000'}}});\n")
backend='/content/ai-knowledge-engine/backend'
def start(args, cwd, log):
    return subprocess.Popen(args,cwd=cwd,env=os.environ.copy(),stdout=open(log,'w'),stderr=subprocess.STDOUT)
api=start(['/content/ake-python312/bin/uvicorn','ake_backend.asgi:app','--host','127.0.0.1','--port','8000'], backend,'/content/ake-api.log')
worker=start(['/content/ake-python312/bin/python','-m','ake_backend.worker'],backend,'/content/ake-worker.log')
ui=start(['npm','run','dev','--','--port','5173'],str(front),'/content/ake-ui.log')
time.sleep(8)
for name, process in [('API',api),('worker',worker),('UI',ui)]:
    if process.poll() is not None: raise RuntimeError(name + ' stopped; inspect /content/ake-*.log')
with urllib.request.urlopen('http://127.0.0.1:5173/api/v1/health/ready', timeout=10) as response:
    print('API readiness:', response.read().decode())
print('Open the interface using the next cell. Keep this runtime connected.')


In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(5173, height=900)


Если интерфейс не отображается, попробуйте открыть блокнот в другом браузере. Среда Colab временная; при перезапуске запускайте ячейки заново. Для генерации ответов требуется отдельно настроить провайдера и согласие на отправку фрагментов документов.
